In [ ]:
# ==========================================
# TESIS ZMVM: PM, clima y salud
# Autor: Arely Leal
# Descripción: Script en Python para el calculo de FAP de material particulado para las 8 am y 6 pm a partir del RR. 
# FUNCIONA PARA PM10 Y PM2.5
# Periodo: 2000-2019
# ==========================================


In [5]:
import pandas as pd
import numpy as np
import os
import math

# Cargar Archivos
FILE_PATH = "AGREGAR RUTA DEL ARCHIVO (RR)"
BASE_DIR = os.path.dirname(os.path.abspath(FILE_PATH)) if os.path.dirname(FILE_PATH) else "."

#Definir funcioes
def redondeo_tesis(x):
    if pd.isna(x): return 0
    entero = int(x)
    decimal = round(x - entero, 10) 
    return math.ceil(x) if decimal > 0.4 else math.floor(x)

def consolidar_categoria(cat):
    cat_str = str(cat).upper()
    if "_EM_" in cat_str: return "EM"
    if "_MM_" in cat_str: return "MM"
    if "_M_" in cat_str: return "M"
    return np.nan

#calculo mensual

df = pd.read_csv(FILE_PATH)
df.columns = df.columns.str.strip()
cols_num = ['Anio', 'Mes', 'hora', 'nx', 'RR']
for c in cols_num:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.dropna(subset=cols_num + ['Enfermedad', 'Sexo', 'Edad_gpo', 'categoria'])
df['Categoria_PM'] = df['categoria'].apply(consolidar_categoria)
df = df.dropna(subset=['Categoria_PM'])

bloque_cols = ['Enfermedad', 'Anio', 'Mes', 'hora', 'Sexo', 'Edad_gpo', 'Categoria_PM']
df['p_i'] = df['nx'] / df.groupby(bloque_cols)['nx'].transform('sum')
df['num_i'] = df['p_i'] * (df['RR'] - 1)
df['den_i'] = df['p_i'] * df['RR']

df_mensual = df.groupby(bloque_cols).agg(
    nx_total_mes=('nx', 'sum'),
    suma_num=('num_i', 'sum'),
    suma_den=('den_i', 'sum')
).reset_index()

df_mensual['FAP_raw'] = (df_mensual['suma_num'] / df_mensual['suma_den']).clip(lower=0)
df_mensual['FAP_%'] = (df_mensual['FAP_raw'] * 100).apply(redondeo_tesis)

df_mensual.drop(columns=['suma_num', 'suma_den', 'FAP_raw']).to_csv(os.path.join(BASE_DIR, "NOMBRE DE SALIDA.csv"), index=False)

#Calculo Anual (8am y 6pm)
df_mensual['fap_pond'] = df_mensual['FAP_raw'] * df_mensual['nx_total_mes']
anual_cols = ['Enfermedad', 'Anio', 'hora', 'Sexo', 'Edad_gpo', 'Categoria_PM']

df_anual = df_mensual.groupby(anual_cols).agg(
    sum_pond=('fap_pond', 'sum'),
    sum_nx=('nx_total_mes', 'sum'),
    meses=('Mes', 'nunique')
).reset_index()

df_anual['FAP_anual_raw'] = df_anual['sum_pond'] / df_anual['sum_nx']
df_anual['FAP_%'] = (df_anual['FAP_anual_raw'] * 100).apply(redondeo_tesis)
df_anual['cobertura_%'] = (df_anual['meses'] / 12 * 100).round(1)

cols_final = ['Enfermedad', 'Anio', 'hora', 'Sexo', 'Edad_gpo', 'Categoria_PM', 'meses', 'cobertura_%', 'FAP_%']
df_anual[cols_final].to_csv(os.path.join(BASE_DIR, "NOMBRE DE SALIDA.csv"), index=False)

print("Archivos generados con éxito.")

Proceso finalizado: Archivos generados con éxito para 8 AM y 6 PM.
